## Make dataset out of Hugginggface dataset

In [ ]:
#pip install --upgrade bitsandbytes transformers

In [ ]:
import os
from tqdm import tqdm
from datasets import load_dataset
import json
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

In [ ]:
dataset = load_dataset("dpdl-benchmark/oxford_flowers102")

train_dataset = dataset["train"]
fid_train_dataset = dataset["test"]
fid_test_dataset = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/test-00000-of-00006.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

data/test-00001-of-00006.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/418M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1020 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6149 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1020 [00:00<?, ? examples/s]

## Load LLaVA Model (4-bit quantized for memory efficiency)

In [ ]:
# Configuration pour quantization 4-bit (économise la VRAM)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Charger LLaVA-1.5-7b
model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"Modèle LLaVA chargé sur: {model.device}")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Modèle LLaVA chargé sur: cuda:0


## Fonction de génération de captions LLaVA

In [ ]:
def generate_llava_caption(image, processor, model, max_new_tokens=256):
    """
    Génère une caption riche et détaillée avec LLaVA.

    Args:
        image: PIL Image
        processor: LLaVA processor
        model: LLaVA model
        max_new_tokens: nombre max de tokens à générer

    Returns:
        str: caption détaillée de l'image
    """
    # Prompt optimisé pour des descriptions riches de fleurs
    prompt = """USER: <image>
Describe this flower image in detail. Include:
- The type/species of flower if recognizable
- Colors (primary and secondary)
- Shape and structure of petals
- Any distinctive features
- The overall aesthetic quality

Provide a rich, detailed description suitable for image generation training.
ASSISTANT:"""

    # Préparation des inputs
    inputs = processor(text=prompt, images=image, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Génération
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    # Décodage et nettoyage
    generated_text = processor.decode(output[0], skip_special_tokens=True)

    # Extraire seulement la réponse de l'assistant
    if "ASSISTANT:" in generated_text:
        caption = generated_text.split("ASSISTANT:")[-1].strip()
    else:
        caption = generated_text.strip()

    return caption

# Test rapide avec une image du dataset
test_image = train_dataset[0]["image"]
test_caption = generate_llava_caption(test_image, processor, model)
print("Caption de test:")
print(test_caption)

Caption de test:
The image features a beautiful white flower with yellow tips, possibly a water lily, floating in a black and white pond. The flower has a circular shape with a prominent yellow center, adding a vibrant touch to the overall aesthetic quality. The petals are large and delicate, creating an elegant appearance. The contrast between the white petals and the yellow tips, as well as the black and white pond, further enhances the visual appeal of this stunning flower.


## Générer le dataset avec captions LLaVA (respecte le split train/fid_train/fid_test)

In [ ]:
# Configuration des répertoires - VERSION LLAVA
root_dir = "./datasets/oxford_flowers102_llava"
images_dir = "images"
captions_dir = "captions"

images_dir_absolute = os.path.join(root_dir, images_dir)
captions_dir_absolute = os.path.join(root_dir, captions_dir)

# Créer la structure de répertoires
os.makedirs(os.path.join(root_dir, images_dir), exist_ok=True)
os.makedirs(os.path.join(root_dir, captions_dir), exist_ok=True)
os.makedirs(os.path.join(root_dir, "partition"), exist_ok=True)

image_format = "png"

def process_split(dataset_split, split_name, start_order=0):
    """
    Traite un split du dataset et génère les captions LLaVA.

    Args:
        dataset_split: le split du dataset (train, test, validation)
        split_name: nom du split pour le fichier JSON
        start_order: index de départ pour la numérotation des images

    Returns:
        tuple: (data_info list, dernier order utilisé)
    """
    data_info = []
    order = start_order

    print(f"\n{'='*50}")
    print(f"Traitement du split: {split_name} ({len(dataset_split)} images)")
    print(f"{'='*50}")

    for item in tqdm(dataset_split, desc=f"Génération captions {split_name}"):
        image = item["image"]

        # Sauvegarder l'image
        image_path = f"{images_dir_absolute}/{order}.{image_format}"
        image.save(image_path)

        # Générer caption LLaVA (riche et détaillée)
        try:
            llava_caption = generate_llava_caption(image, processor, model)
        except Exception as e:
            print(f"Erreur sur image {order}: {e}")
            # Fallback sur le label simple en cas d'erreur
            llava_caption = train_dataset.features["label"].int2str(item["label"])

        # Sauvegarder la caption
        with open(f"{captions_dir_absolute}/{order}.txt", "w") as text_file:
            text_file.write(llava_caption)

        width, height = image.size
        ratio = width / height

        data_info.append({
            "height": height,
            "width": width,
            "ratio": ratio,
            "path": f"images/{order}.{image_format}",
            "prompt": llava_caption,
            "split": split_name
        })

        order += 1

    return data_info, order

# ============================================================
# GÉNÉRATION DES SPLITS AVEC CAPTIONS LLAVA (SEULMENT TRAIN)
# ============================================================

all_data_info = []
current_order = 0

# 1. Train split (pour fine-tuning)
train_info, current_order = process_split(train_dataset, "train", current_order)
all_data_info.extend(train_info)

# # 2. FID Train split (test set du HF dataset) - COMMENTÉ
# fid_train_info, current_order = process_split(fid_train_dataset, "fid_train", current_order)
# all_data_info.extend(fid_train_info)

# # 3. FID Test split (validation set du HF dataset) - COMMENTÉ
# fid_test_info, current_order = process_split(fid_test_dataset, "fid_test", current_order)
# all_data_info.extend(fid_test_info)

# ============================================================
# SAUVEGARDER LES FICHIERS JSON PAR SPLIT
# ============================================================

# JSON global avec toutes les données (ici, seulement train)
with open(os.path.join(root_dir, "partition/data_info.json"), "w") as f:
    json.dump(all_data_info, f, indent=2)

# JSON séparés par split (utile pour le training)
with open(os.path.join(root_dir, "partition/train.json"), "w") as f:
    json.dump(train_info, f, indent=2)

# # JSON pour FID Train - COMMENTÉ
# with open(os.path.join(root_dir, "partition/fid_train.json"), "w") as f:
#     json.dump(fid_train_info, f, indent=2)

# # JSON pour FID Test - COMMENTÉ
# with open(os.path.join(root_dir, "partition/fid_test.json"), "w") as f:
#     json.dump(fid_test_info, f, indent=2)

print(f"\n{'='*50}")
print("RÉSUMÉ DE LA GÉNÉRATION")
print(f"{'='*50}")
print(f"Train: {len(train_info)} images")
# print(f"FID Train: {len(fid_train_info)} images") # COMMENTÉ
# print(f"FID Test: {len(fid_test_info)} images") # COMMENTÉ
print(f"Total: {len(all_data_info)} images")
print(f"\nDataset sauvegardé dans: {root_dir}")


Traitement du split: train (1020 images)


Génération captions train: 100%|██████████| 1020/1020 [2:08:38<00:00,  7.57s/it]


RÉSUMÉ DE LA GÉNÉRATION
Train: 1020 images
Total: 1020 images

Dataset sauvegardé dans: ./datasets/oxford_flowers102_llava
